In [34]:
#print("Starting script")
import torch
#print("Imported torch")
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
#print("imported torchvision")
from torch.utils.data import DataLoader
import torchbnn as bnn

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [35]:
# MNIST data
transform = transforms.ToTensor()

train_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)


In [36]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [37]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=3
        )

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)

        self.flatten = nn.Flatten()

        self.mlp = nn.Sequential(
            bnn.BayesLinear(prior_mu=0, prior_sigma=0.1, in_features = 8 * 13 * 13, out_features = 64),
            nn.Tanh(),
            bnn.BayesLinear(prior_mu=0, prior_sigma=0.1, in_features=64, out_features=32),
            nn.Tanh(),
            bnn.BayesLinear(prior_mu=0, prior_sigma=0.1, in_features=32, out_features=10),
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.mlp(x)
        return x


In [41]:
model = CNN().to(device)

loss_fn = nn.CrossEntropyLoss()
kl_loss = bnn.BKLLoss(reduction = 'mean', last_layer_only = False) # kl divergence loss
kl_weight = 1.0 / len(train_loader)
optimizer = optim.SGD(model.parameters(), lr=0.05)

epochs = 5

for epoch in range(epochs):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        # Forward pass
        output = model(X)
        cross_entropy = loss_fn(output, y)
        kl = kl_loss(model)
        loss = cross_entropy + kl_weight*kl

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(output, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    acc = correct / total
    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch}: Loss = {avg_loss:.4f}, Accuracy = {acc:.4f}")

Epoch 0: Loss = 1.2502, Accuracy = 0.5687
Epoch 1: Loss = 0.4305, Accuracy = 0.8639
Epoch 2: Loss = 0.2861, Accuracy = 0.9113
Epoch 3: Loss = 0.2253, Accuracy = 0.9306
Epoch 4: Loss = 0.1926, Accuracy = 0.9412


In [42]:
# Test accuracy
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        outputs = []  # using multiple forward passes instead of one

        for _ in range(20):
            outputs.append(model(X))

        output = torch.mean(torch.stack(outputs), dim=0)
        
        preds = torch.argmax(output, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

test_acc = correct / total
print("Test Accuracy:", test_acc)

Test Accuracy: 0.9673
